In [ ]:
"""RQ2 post-inference evaluation.

Run `evaluate_rq2_v2.ipynb` from the Hub project root after inference.
Alternatively, convert the percent-separated source into a notebook.
This evaluator
loads no model, makes no network requests, and never uses a final-project split.
Only the pinned local tokenizer is loaded on CPU to verify template token counts.
Dependencies: Python 3.10+ and the tokenizers package used by the inference run.
It requires complete inference artifacts; it does not wait, resume, or poll.
Outputs go to `results/rq2_h100_v2/evaluation/`. User labels are cohort positions,
not customer IDs. Audit judgments and the student's hand check remain unperformed.
"""

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import hashlib
import importlib.util
import json
import math
import os
import re
import sys
import time
import zipfile

os.environ['CUDA_VISIBLE_DEVICES'] = ''
PROJECT = Path.cwd()
DATA = PROJECT / 'data/inputs_v1'
RESULTS = PROJECT / 'results/rq2_h100_v2'
OUT = RESULTS / 'evaluation'
PROTOCOL_HASH = 'd46b45deaf2c2f955c52fa020082971947d5ac8e27073a90d1c5384f648af697'
ORDER_HASH = '8047bbccbbe66f0bc420df36b06484c4f0888fe18deb8caf0897f1d4903acb04'
MANIFEST_HASH = '2d1669cc6d970409b0706178a6a83f6c2f2720ab7421aa7665d7685529322b12'
REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
BASELINES = ('popularity', 'cbf', 'latest_buy', 'latest_buy_or_sell')
VARIANTS = ('raw', 'summary', 'template')
AUDIT_POSITIONS = (1, 2, 3, 25, 50, 75, 100, 101, 125, 150, 175, 200)
PAIRS = (('summary', 'raw'), ('template', 'raw'), ('summary', 'template'))
DISCOUNTS = tuple(1 / math.log2(rank + 2) for rank in range(10))


def require(condition, message):
    if not condition:
        raise ValueError(message)  # No row contents or customer IDs in diagnostics.


def sha(path):
    with path.open('rb') as stream:
        result = hashlib.sha256()
        for block in iter(lambda: stream.read(4 * 1024**2), b''):
            result.update(block)
        return result.hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


def read_jsonl(path):
    with path.open(encoding='utf-8') as stream:
        return [json.loads(line) for line in stream if line.strip()]


def read_csv(path):
    with path.open(encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def close(a, b):
    return math.isclose(a, b, rel_tol=0, abs_tol=1e-12)


def ndcg(ranking, truth):
    """Independent binary scorer; IDCG uses all eligible truth, not pool truth."""
    if not truth:
        return 0.0
    return sum(d for asset, d in zip(ranking[:10], DISCOUNTS) if asset in truth) / sum(DISCOUNTS[:min(10, len(truth))])


def mean(values):
    return sum(values) / len(values) if values else None


def describe(values):
    return {'n': len(values), 'mean': mean(values),
            'min': min(values) if values else None, 'max': max(values) if values else None}


def messages(prompt):
    return [{'role': 'system', 'content': prompt[0]}, {'role': 'user', 'content': prompt[1]}]


def validate_positions(rows, label):
    require(len(rows) == 200 and [r['position'] for r in rows] == list(range(1, 201)),
            label + ': require all 200 rows in frozen order')


def parsed_matched(text, candidates, shared):
    # Reproduce inference parsing and fallback, then cross-check saved predictions.
    if text is None:
        return candidates[:10], False, True
    try:
        numbers = shared.parse_ranking(text, len(candidates))
    except TypeError:
        numbers = []
    if len(numbers) != 10:
        return candidates[:10], False, True
    try:
        native = json.loads(text)
    except json.JSONDecodeError:
        native = None
    exact = isinstance(native, dict) and native.get('ranking') == numbers and set(native) == {'ranking'}
    return [candidates[n - 1] for n in numbers], not exact, False


def template_text(ctx, encode):
    """Independent replay of the notebook's round-robin, whole-entry packing."""
    visible = ctx['history'][:20]
    prefix = f"Among the {len(visible)} shown assets, the investor made {sum(a['buy_count'] for a in visible)} Buy transactions."
    groups = []
    for field, label in [('assetCategory', 'Category by Buy count'), ('marketID', 'Market by Buy count')]:
        counts = Counter()
        for asset in visible:
            value = next((a.split('=', 1)[1] for a in asset['attributes'] if a.startswith(field + '=')), 'unknown')
            counts[value] += asset['buy_count']
        groups.append((label, [f'{key}: {count}' for key, count in sorted(counts.items(), key=lambda p: (-p[1], p[0]))]))
    groups.append(('Top ISINs by Buy count', [f"{a['id']}: {a['buy_count']}" for a in visible]))
    kept = [[] for _ in groups]

    def render():
        return ' '.join([prefix] + [f'{label}: {", ".join(entries)}.' for (label, _), entries in zip(groups, kept) if entries])

    for index in range(max(len(entries) for _, entries in groups)):
        for group, (_, entries) in enumerate(groups):
            if index < len(entries):
                kept[group].append(entries[index])
                if len(encode(render())) > 120:
                    kept[group].pop()
    require(len(encode(render())) <= 120, 'Template token budget exceeded')
    return render()

In [ ]:
def verify_inputs(data=DATA):
    require(sha(data / 'input-manifest.json') == MANIFEST_HASH, 'Unrecognized input manifest')
    manifest = read_json(data / 'input-manifest.json')
    for name, expected in manifest.items():
        path = (data / name).resolve()
        require(path.is_relative_to(data.resolve()), 'Unsafe manifest path')
        require(sha(path) == expected, 'Frozen input hash mismatch: ' + name)
    protocol = read_json(data / 'starter200/protocol.json')
    payload = {k: v for k, v in protocol.items() if k != 'protocol_hash'}
    require(protocol['protocol_hash'] == PROTOCOL_HASH and
            hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest() == PROTOCOL_HASH,
            'Starter protocol hash mismatch')
    expected = {'cutoff': '2021-06-01', 'end_exclusive': '2021-07-01', 'exclude_seen': False,
                'cohort_selection': 'warm', 'candidate_count': 50, 'k': 10,
                'initial_positions': 'excluded', 'duplicate_policy': 'exact full rows only'}
    require(all(protocol.get(k) == v for k, v in expected.items()), 'Starter protocol settings mismatch')
    contexts = read_jsonl(data / 'starter200/contexts.jsonl')
    users = protocol['users']
    require(len(users) == len(set(users)) == len(contexts) == 200 and
            [c['user_id'] for c in contexts] == users, 'Cohort order/count mismatch')
    candidates = [[a['id'] for a in c['candidate_assets']] for c in contexts]
    require(all(len(c) == len(set(c)) == 50 for c in candidates), 'Invalid frozen candidate pools')
    require(hashlib.sha256('\n'.join(','.join(c) for c in candidates).encode()).hexdigest() == ORDER_HASH,
            'Candidate order digest mismatch')
    shared = load_module('rq2_shared_verified', data / 'shared_rerank.py')
    for context in contexts:
        require(context['cutoff'] == protocol['cutoff'], 'Context cutoff mismatch')
        shared.assert_no_future(context)
    # Extract only the four starter source CSVs. Never use another split or labels file.
    source_dir = data / 'far-trans-data/FAR-Trans'
    with zipfile.ZipFile(data / 'FAR-Trans.zip') as archive:
        for name, expected_hash in protocol['source_sha256'].items():
            blob = archive.read('FAR-Trans/' + name)
            require(hashlib.sha256(blob).hexdigest() == expected_hash, 'Archived source hash mismatch: ' + name)
            path = source_dir / name
            if not path.exists():
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_bytes(blob)
            require(sha(path) == expected_hash, 'Extracted source hash mismatch: ' + name)
    starter = load_module('rq2_starter_verified', data / 'course/far-trans/starter.py')
    assets, profiles, history, future, end, excluded = starter.load_data(source_dir, protocol['cutoff'], 30)
    require(end == protocol['end_exclusive'], 'Label horizon mismatch')
    require(starter.select_users(profiles, history, 'warm', 200, None) == users, 'Deterministic starter cohort mismatch')
    require(sorted(assets) == protocol['eligible_assets'], 'Eligible asset universe mismatch')
    features = {isin: starter.tokens(row) for isin, row in assets.items()}
    pop = starter.popularity_scores(history, assets)
    rebuilt_contexts, rebuilt_ranks = starter.contexts_for(users, assets, profiles, history, features, pop,
                                                         protocol['cutoff'], 50, False)
    # Starter floating-point accumulation iterates sets. A fresh Python hash seed
    # can perturb near-tied CBF order, even under the identical protocol hash.
    # The pinned ordered snapshot is authoritative, as required by PROTOCOL.md.
    regenerated_order_differences = 0
    for uid, context, pool in zip(users, contexts, candidates):
        rebuilt = rebuilt_contexts[uid]
        require({k: v for k, v in rebuilt.items() if k != 'candidate_assets'} ==
                {k: v for k, v in context.items() if k != 'candidate_assets'},
                'Rebuilt non-candidate context fields differ from snapshot')
        regenerated_order_differences += rebuilt['candidate_assets'] != context['candidate_assets']
        cbf_scores = starter.content_scores(history[uid], features)
        if not any(cbf_scores.values()):
            cbf_scores = pop
        for asset in context['candidate_assets']:
            isin = asset['id']
            require(isin in assets and asset == {'id': isin, 'name': assets[isin]['assetName'],
                    'attributes': features[isin], 'metadata_timestamp': assets[isin]['timestamp']},
                    'Frozen candidate metadata differs from starter sources')
        require(all(cbf_scores[a] + 1e-12 >= cbf_scores[b] for a, b in zip(pool, pool[1:])),
                'Frozen pool is not descending CBF score within floating-point tolerance')
        outside = set(context['eligible_asset_ids']) - set(pool)
        require(not outside or cbf_scores[pool[-1]] + 1e-12 >= max(cbf_scores[a] for a in outside),
                'Frozen pool excludes a strictly higher-scoring candidate')
    events = starter.scan_recent_events(source_dir, protocol['cutoff'], assets)
    recent = starter.recency_rankings(rebuilt_contexts, pop, events)
    for uid in users:
        rebuilt_ranks[uid].update(recent[uid])
    truth = [future.get(u, set()) & set(c['eligible_asset_ids']) for u, c in zip(users, contexts)]
    # Read saved baseline predictions with strict user/method/rank cardinality checks.
    saved = defaultdict(dict)
    for row in read_csv(data / 'starter200/predictions.csv'):
        key = (row['user_id'], row['method'])
        rank = int(row['rank'])
        require(key[0] in users and key[1] in BASELINES and rank in range(1, 11), 'Unexpected baseline prediction key')
        require(rank not in saved[key], 'Duplicate baseline prediction rank')
        saved[key][rank] = row['ISIN']
    require(len(saved) == 200 * len(BASELINES), 'Missing baseline predictions')
    rankings = {m: [] for m in BASELINES}
    for uid, context in zip(users, contexts):
        for method in BASELINES:
            ranks = saved[(uid, method)]
            require(set(ranks) == set(range(1, 11)), 'Incomplete baseline ranking')
            ranking = [ranks[r] for r in range(1, 11)]
            starter.validate_ranking(ranking, context['eligible_asset_ids'])
            expected_ranking = [a['id'] for a in context['candidate_assets'][:10]] if method == 'cbf' else rebuilt_ranks[uid][method]
            require(ranking == expected_ranking, 'Baseline ranking differs from frozen CBF control or rebuilt non-CBF prediction')
            rankings[method].append(ranking)
    scores = {m: [ndcg(r, t) for r, t in zip(rankings[m], truth)] for m in BASELINES}
    per_user = {}
    for row in read_csv(data / 'starter200/per_user.csv'):
        key = (row['user_id'], row['method'])
        require(key not in per_user, 'Duplicate frozen per-user score')
        per_user[key] = row
    require(set(per_user) == {(u, m) for u in users for m in BASELINES}, 'Frozen score coverage mismatch')
    metrics = read_json(data / 'starter200/metrics.json')
    require(metrics['protocol_hash'] == PROTOCOL_HASH and metrics['excluded_rows'] == excluded,
            'Frozen metrics provenance mismatch')
    require(set(metrics['methods']) == set(BASELINES), 'Unexpected frozen baseline methods')
    for method in BASELINES:
        for i, uid in enumerate(users):
            row = per_user[(uid, method)]
            require(int(row['relevant_count']) == len(truth[i]) and close(float(row['ndcg_at_10']), scores[method][i]),
                    'Independent baseline score differs from frozen per-user score')
        aggregate = metric_summary(scores[method], rankings[method], truth, list(range(200)))
        require(all(close(aggregate[k], value) if isinstance(value, float) else aggregate[k] == value
                    for k, value in metrics['methods'][method].items()), 'Independent baseline aggregate mismatch')
    return dict(manifest=manifest, protocol=protocol, contexts=contexts, candidates=candidates,
                users=users, truth=truth, rankings=rankings, scores=scores, history=history,
                assets=assets, profiles=profiles, events=events, starter=starter, shared=shared,
                excluded=excluded, source_dir=source_dir,
                regenerated_order_differences=regenerated_order_differences)


def metric_summary(values, rankings, truth, indices):
    nonempty = [i for i in indices if truth[i]]
    hits = [len(set(rankings[i]) & truth[i]) for i in indices]
    return {'users': len(indices), 'nonempty_truth_users': len(nonempty),
            'empty_truth_users': len(indices) - len(nonempty),
            'ndcg_at_10': mean([values[i] for i in indices]),
            'ndcg_at_10_nonempty': mean([values[i] for i in nonempty]),
            'users_with_top10_hit': sum(h > 0 for h in hits),
            'hit_coverage_all': mean([int(h > 0) for h in hits]),
            'hit_coverage_nonempty': mean([int(bool(set(rankings[i]) & truth[i])) for i in nonempty])}

In [ ]:
def verify_inference(bundle, results=RESULTS, project=PROJECT):
    # All paths are opened once, only when explicitly run after inference completes.
    required = ['model.json', 'template-generation.json', 'llm-summary.jsonl', 'llm-summary-runtime.json']
    required += [f'{v}{suffix}' for v in (*VARIANTS, 'provided')
                 for suffix in ('.jsonl', '-runtime.json', '-rankings.json')]
    require(all((results / name).is_file() for name in required), 'Inference artifacts incomplete; run evaluator only after completion')
    inputs_hashes = {name: sha(results / name) for name in required}
    model = read_json(results / 'model.json')
    expected = {'model': 'Qwen/Qwen3-1.7B', 'revision': REVISION, 'dtype': 'torch.float32',
                'tf32': False, 'do_sample': False, 'thinking': False, 'seed': 0, 'batch_size': 8,
                'attention': 'sdpa', 'physical_gpu': 3, 'visible_gpus': 1,
                'protocol_hash': PROTOCOL_HASH, 'candidate_order_sha256': ORDER_HASH}
    require(all(model.get(k) == v for k, v in expected.items()) and 'H100' in model['gpu'], 'Inference configuration mismatch')
    for name in ('torch', 'torch_cuda', 'transformers', 'python', 'os', 'load_seconds',
                 'data_preparation_seconds', 'checkpoint_wait_and_verification_seconds'):
        require(name in model, 'Missing environment/time metadata: ' + name)
    offline = Path('/home/rshangareevil/REC-23097-20261116/llm4rec_s2_rq2/offline_packages_v1')
    if offline.is_dir():
        sys.path.insert(0, str(offline))
    # Tokenizer only: no transformers model, torch, GPU, or remote loading.
    from tokenizers import Tokenizer
    model_dir = Path('/tmp/llm4rec-session3-rq2/models/Qwen3-1.7B-70d244cc')
    tree = {x['path']: x for x in read_json(DATA / 'hf-tree.json') if x['type'] == 'file'}
    require(read_json(DATA / 'hf-api.json')['sha'] == REVISION, 'Tokenizer revision mismatch')
    token_path = model_dir / 'tokenizer.json'
    entry = tree['tokenizer.json']
    require(token_path.stat().st_size == entry['size'], 'Tokenizer size mismatch')
    if 'lfs' in entry:
        require(sha(token_path) == entry['lfs']['oid'], 'Tokenizer hash mismatch')
    else:
        require(hashlib.sha1(f'blob {entry["size"]}\0'.encode() + token_path.read_bytes()).hexdigest() == entry['oid'],
                'Tokenizer git hash mismatch')
    tokenizer = Tokenizer.from_file(str(token_path))
    encode = lambda text: tokenizer.encode(text, add_special_tokens=False).ids
    template = read_json(results / 'template-generation.json')
    require(len(template['summaries']) == 200 and template['wall_seconds'] >= 0, 'Template artifact incomplete')
    contexts, candidates, shared = bundle['contexts'], bundle['candidates'], bundle['shared']
    require(template['summaries'] == [template_text(c, encode) for c in contexts], 'Template differs from independently replayed packing')
    sys.path.insert(0, str(DATA / 'course/far-trans-llm'))
    provided = load_module('rq2_provided_verified', DATA / 'course/far-trans-llm/ranker.py')
    stages, runtimes, failures = {}, {}, {}
    for stage in ('llm-summary', *VARIANTS, 'provided'):
        rows = read_jsonl(results / f'{stage}.jsonl')
        validate_positions(rows, stage)
        budget = 120 if stage == 'llm-summary' else 512 if stage == 'provided' else 60
        for row in rows:
            require(type(row['input_tokens']) is int and row['input_tokens'] > 0 and
                    type(row['output_tokens']) is int and 0 <= row['output_tokens'] <= budget and
                    type(row['hit_token_cap']) is bool, 'Invalid logged token counts')
            error = row['input_tokens'] + budget > 8192
            require(row['error'] == ('context_limit_exceeded' if error else None), 'Context-limit status mismatch')
            require((row['text'] is None and row['output_tokens'] == 0 and not row['hit_token_cap']) if error
                    else (isinstance(row['text'], str) and row['output_tokens'] > 0), 'Generation record inconsistent')
            require(not row['hit_token_cap'] or row['output_tokens'] == budget, 'Token-cap status inconsistent')
        runtime = read_json(results / f'{stage}-runtime.json')
        totals = {'requests': 200, 'max_new_tokens': budget,
                  'input_tokens': sum(r['input_tokens'] for r in rows), 'output_tokens': sum(r['output_tokens'] for r in rows),
                  'token_cap_count': sum(r['hit_token_cap'] for r in rows), 'context_limit_errors': sum(bool(r['error']) for r in rows)}
        require(all(runtime[k] == v for k, v in totals.items()) and runtime['wall_seconds'] >= 0, 'Stage runtime totals mismatch')
        stages[stage], runtimes[stage] = rows, runtime
    for i, ctx in enumerate(contexts):
        require(stages['llm-summary'][i]['messages'] == messages(shared.summary_prompt(ctx, 'C', {})), 'Summary prompt differs from actual raw history')
        blocks = {'raw': shared.history_block(ctx), 'summary': stages['llm-summary'][i]['text'] or '[No summary returned.]',
                  'template': template['summaries'][i]}
        for variant in VARIANTS:
            require(stages[variant][i]['messages'] == messages(shared.rank_prompt(ctx, blocks[variant])), 'Matched rank prompt mismatch: ' + variant)
        prompt = provided.build_prompt(ctx, bundle['assets'], bundle['history'][ctx['user_id']],
                                       bundle['events'].get(ctx['user_id'], ()), history_events=15, shuffle_seed=None)
        require(stages['provided'][i]['messages'] == [{'role': 'user', 'content': prompt}], 'Supplied ranker prompt mismatch')
    for variant in (*VARIANTS, 'provided'):
        predictions = read_json(results / f'{variant}-rankings.json')
        validate_positions(predictions, variant + ' predictions')
        for row, prediction, pool in zip(stages[variant], predictions, candidates):
            if variant == 'provided':
                repair, fallback = False, False
                try:
                    rank, repair = provided.parse_ranking(row['text'], set(pool))
                    bundle['starter'].validate_ranking(rank, pool)
                except (ValueError, TypeError, KeyError, RuntimeError, AttributeError):
                    rank, fallback = pool[:10], True
            else:
                rank, repair, fallback = parsed_matched(row['text'], pool, shared)
            require(type(prediction['repair']) is bool and type(prediction['fallback']) is bool and
                    (prediction['ranking'], prediction['repair'], prediction['fallback']) == (rank, repair, fallback),
                    'Saved prediction/repair/fallback disagrees with independent parser replay: ' + variant)
            bundle['starter'].validate_ranking(rank, pool)
        bundle['rankings'][variant] = [p['ranking'] for p in predictions]
        bundle['scores'][variant] = [ndcg(r, t) for r, t in zip(bundle['rankings'][variant], bundle['truth'])]
        failures[variant] = {'repairs': sum(p['repair'] for p in predictions), 'fallbacks': sum(p['fallback'] for p in predictions),
                             'token_caps': runtimes[variant]['token_cap_count'], 'context_errors': runtimes[variant]['context_limit_errors'],
                             'empty_text': sum(r['text'] == '' for r in stages[variant]),
                             'nonfallback_token_caps': sum(r['hit_token_cap'] and not p['fallback'] for r, p in zip(stages[variant], predictions))}
        stages[variant + '-predictions'] = predictions
    require(inputs_hashes == {name: sha(results / name) for name in required}, 'Inference files changed during evaluation')
    return dict(model=model, stages=stages, runtimes=runtimes, failures=failures, template=template,
                artifact_sha256=inputs_hashes)

In [ ]:
def analyze(bundle, inference):
    truth, scores, rankings = bundle['truth'], bundle['scores'], bundle['rankings']
    stages, pools = inference['stages'], bundle['candidates']
    sparse = [i for i, c in enumerate(bundle['contexts']) if 1 <= sum(a['buy_count'] for a in c['history']) <= 5]
    nonempty = [i for i in range(200) if truth[i]]
    reachable = [len(set(pool) & t) for pool, t in zip(pools, truth)]
    oracle = [sum(DISCOUNTS[:min(10, n)]) / sum(DISCOUNTS[:min(10, len(t))]) if t else 0.0 for n, t in zip(reachable, truth)]
    paired = {}
    for a, b in PAIRS:
        delta = [x - y for x, y in zip(scores[a], scores[b])]
        def group(indices):
            return {**describe([delta[i] for i in indices]),
                    'improved': sum(delta[i] > 1e-12 for i in indices),
                    'worsened': sum(delta[i] < -1e-12 for i in indices),
                    'unchanged': sum(abs(delta[i]) <= 1e-12 for i in indices)}
        paired[a + '_minus_' + b] = {'all': group(list(range(200))), 'nonempty_truth': group(nonempty), 'sparse': group(sparse)}
    per_user = []
    for i, (ctx, t) in enumerate(zip(bundle['contexts'], truth)):
        row = {'user': f'U{i + 1:03d}', 'position': i + 1, 'truth_count': len(t),
               'eligible_history_buys': sum(a['buy_count'] for a in ctx['history']), 'sparse': i in sparse,
               'visible_assets': min(20, len(ctx['history'])), 'total_history_assets': len(ctx['history']),
               'reachable_truth_count': reachable[i], 'unreachable_truth_count': len(t) - reachable[i],
               'candidate_oracle_ndcg': oracle[i], 'summary_missing': not bool(stages['llm-summary'][i]['text']),
               'summary_token_cap': stages['llm-summary'][i]['hit_token_cap'],
               'summary_context_error': bool(stages['llm-summary'][i]['error'])}
        for method in rankings:
            hits = len(set(rankings[method][i]) & t)
            row[method + '_ndcg'] = scores[method][i]
            row[method + '_hits'] = hits
            if method in (*VARIANTS, 'provided'):
                pred = stages[method + '-predictions'][i]
                row.update({method + '_repair': pred['repair'], method + '_fallback': pred['fallback'],
                            method + '_token_cap': stages[method][i]['hit_token_cap'],
                            method + '_context_error': bool(stages[method][i]['error']),
                            method + '_reachable_missed': reachable[i] - hits,
                            method + '_oracle_gap': oracle[i] - scores[method][i]})
        for a, b in PAIRS:
            row[a + '_minus_' + b] = scores[a][i] - scores[b][i]
            row[a + '_vs_' + b + '_same_order'] = rankings[a][i] == rankings[b][i]
            row[a + '_vs_' + b + '_top10_overlap'] = len(set(rankings[a][i]) & set(rankings[b][i]))
        per_user.append(row)
    reachability = {'nonempty_truth_users': len(nonempty), 'users_with_reachable_truth': sum(n > 0 for n in reachable),
                    'nonempty_users_with_no_reachable_truth': sum(reachable[i] == 0 for i in nonempty),
                    'nonempty_users_with_partly_missing_truth': sum(0 < reachable[i] < len(truth[i]) for i in nonempty),
                    'total_user_truth_pairs': sum(map(len, truth)), 'reachable_user_truth_pairs': sum(reachable),
                    'hit_coverage_ceiling_all': mean([int(n > 0) for n in reachable]),
                    'hit_coverage_ceiling_nonempty': mean([int(reachable[i] > 0) for i in nonempty]),
                    'mean_truth_recall_in_pool_nonempty': mean([reachable[i] / len(truth[i]) for i in nonempty]),
                    'oracle_ndcg_all': mean(oracle), 'oracle_ndcg_nonempty': mean([oracle[i] for i in nonempty])}
    changes = {}
    for a, b in PAIRS:
        key = a + '_minus_' + b
        groups = {'improved': [r for r in per_user if r[key] > 1e-12],
                  'worsened': [r for r in per_user if r[key] < -1e-12],
                  'unchanged': [r for r in per_user if abs(r[key]) <= 1e-12]}
        changes[key] = {name: {'users': len(rows), 'same_top10_order': sum(r[a + '_vs_' + b + '_same_order'] for r in rows),
                              'any_fallback': sum(r[a + '_fallback'] or r[b + '_fallback'] for r in rows),
                              'any_repair': sum(r[a + '_repair'] or r[b + '_repair'] for r in rows),
                              'summary_missing': sum(r['summary_missing'] for r in rows),
                              'summary_capped': sum(r['summary_token_cap'] for r in rows),
                              'empty_truth': sum(r['truth_count'] == 0 for r in rows),
                              'nonempty_no_reachable_truth': sum(r['truth_count'] > 0 and r['reachable_truth_count'] == 0 for r in rows)}
                        for name, rows in groups.items()}
    for method in (*VARIANTS, 'provided'):
        inference['failures'][method].update(
            reachable_users_with_no_top10_hit=sum(reachable[i] > 0 and not (set(rankings[method][i]) & truth[i]) for i in range(200)),
            reachable_truth_omitted_from_top10=sum(reachable[i] - len(set(rankings[method][i]) & truth[i]) for i in range(200)),
            mean_gap_to_pool_oracle=mean([oracle[i] - scores[method][i] for i in range(200)]))
    summary = {'protocol_hash': PROTOCOL_HASH, 'candidate_order_sha256': ORDER_HASH,
               'users': 200, 'nonempty_truth_users': len(nonempty), 'empty_truth_users': 200 - len(nonempty),
               'matched': {m: metric_summary(scores[m], rankings[m], truth, list(range(200))) for m in VARIANTS},
               'starter_baselines': {m: metric_summary(scores[m], rankings[m], truth, list(range(200))) for m in BASELINES},
               'supplied_ranker_separate_comparison': metric_summary(scores['provided'], rankings['provided'], truth, list(range(200))),
               'sparse_1_to_5_buys': {'users': len(sparse), 'nonempty_truth_users': sum(bool(truth[i]) for i in sparse),
                                    'users_with_reachable_truth': sum(reachable[i] > 0 for i in sparse),
                                    'methods': {m: metric_summary(scores[m], rankings[m], truth, sparse) for m in rankings},
                                    'oracle_ndcg': mean([oracle[i] for i in sparse])},
               'paired': paired, 'changes': changes, 'reachability': reachability,
               'failures': inference['failures'],
               'summary_generation': {'missing_or_empty': sum(r['summary_missing'] for r in per_user),
                                      **inference['runtimes']['llm-summary']},
               'model_and_environment': inference['model'], 'stage_runtimes': inference['runtimes'],
               'timing': {'data_preparation_seconds': inference['model']['data_preparation_seconds'],
                          'checkpoint_wait_and_verification_seconds': inference['model']['checkpoint_wait_and_verification_seconds'],
                          'model_load_seconds': inference['model']['load_seconds'],
                          'template_generation_seconds': inference['template']['wall_seconds'],
                          'matched_inference_stage_wall_seconds': sum(inference['runtimes'][s]['wall_seconds'] for s in ('llm-summary', *VARIANTS)),
                          'provided_inference_stage_wall_seconds': inference['runtimes']['provided']['wall_seconds'],
                          'pre_hub_offline_preparation_seconds': None,
                          'timing_note': 'Stage wall time includes tokenization, generation, logging; excludes between-stage prompt construction. Pre-Hub download/packaging time is unmeasured. No end-to-end inference wall time was logged.'},
               'verification': {'all_frozen_hashes_verified': True, 'source_sha256': bundle['protocol']['source_sha256'],
                                'independent_frozen_baseline_match': True, 'noncandidate_contexts_rebuilt_exactly': True,
                                'candidate_metadata_verified': True,
                                'fresh_starter_CBF_order_differences': bundle['regenerated_order_differences'],
                                'cbf_order_policy': 'Pinned ordered snapshot is authoritative; freshly regenerated near-tie order is diagnostic only.',
                                'prompts_and_predictions_replayed': True, 'template_replayed_with_pinned_tokenizer': True,
                                'inference_artifact_sha256': inference['artifact_sha256']},
               'interpretation': ['Development purchase-label relevance only; repeated Buys allowed; all 200 retained.',
                                  'Fresh starter CBF near-tie order can vary with Python hash seed; the verified frozen candidate order and baseline predictions are authoritative.',
                                  'Supplied ranker is separate: profile, names, dates, full counts, recent events and 512 output tokens.',
                                  'Raw vs summary changes compression procedure as well as wording. Template uses visible rows only.',
                                  'Template greedily packs whole entries round-robin; a rejected entry can be followed by a shorter accepted one.',
                                  'Same planned batches of eight; context rejection can change effective generation batch membership.',
                                  'Capped output is a truncation flag, not automatically an invalid ranking or hallucination.',
                                  'Summary audits are pending human review; omission alone is not hallucination.',
                                  'No final-project split, investment suitability, or benchmark placement claim.'],
               'student_hand_check': {'status': 'reserved_unperformed', 'attributed_to_student': False}}
    return summary, per_user

In [ ]:
def prepare_audits(bundle, inference, per_user):
    scores, truth, rankings = bundle['scores'], bundle['truth'], bundle['rankings']
    selected = set(AUDIT_POSITIONS)
    reasons = defaultdict(list)
    for position in AUDIT_POSITIONS:
        reasons[position].append('protocol_position')
    for a, b in PAIRS:
        delta = [x - y for x, y in zip(scores[a], scores[b])]
        for label, indices in [('strongest_improvement', [i for i in range(200) if delta[i] > 1e-12]),
                               ('strongest_worsening', [i for i in range(200) if delta[i] < -1e-12])]:
            if indices:
                strongest = max(abs(delta[i]) for i in indices)
                # Include all tied strongest changes; never select an unchanged user as a change.
                for i in indices:
                    if close(abs(delta[i]), strongest):
                        selected.add(i + 1)
                        reasons[i + 1].append(a + '_minus_' + b + ':' + label)
    examples = []
    for position in sorted(selected):
        i = position - 1
        ctx = bundle['contexts'][i]
        visible = ctx['history'][:20]
        counts = {}
        for field in ('assetCategory', 'marketID'):
            counter = Counter()
            for asset in visible:
                value = next((v.split('=', 1)[1] for v in asset['attributes'] if v.startswith(field + '=')), 'unknown')
                counter[value] += asset['buy_count']
            counts[field] = dict(sorted(counter.items(), key=lambda p: (-p[1], p[0])))
        generated = inference['stages']['llm-summary'][i]
        actual_input = generated['messages'][1]['content']
        text = generated['text'] or ''
        # Exact ISIN strings absent from the input are review flags, not automatic hallucination judgments.
        isin_flags = sorted(set(re.findall(r'\b[A-Z]{2}[A-Z0-9]{9}[0-9]\b', text)) -
                            set(re.findall(r'\b[A-Z]{2}[A-Z0-9]{9}[0-9]\b', actual_input)))
        examples.append({'user': f'U{position:03d}', 'position': position, 'selection_reasons': reasons[position],
                         'actual_summary_messages': generated['messages'], 'actual_history_block': bundle['shared'].history_block(ctx),
                         'generated_summary': generated['text'], 'template_summary': inference['template']['summaries'][i],
                         'visible_evidence': {'assets': len(visible), 'buys': sum(a['buy_count'] for a in visible),
                                              'repeated_assets': sum(a['buy_count'] > 1 for a in visible), 'buy_weighted_counts': counts},
                         'history_assets_omitted_by_input_cap': max(0, len(ctx['history']) - 20),
                         'summary_hit_token_cap': generated['hit_token_cap'], 'summary_error': generated['error'],
                         'ISIN_strings_absent_from_actual_input_review_flags': isin_flags,
                         'paired_scores': {a + '_minus_' + b: scores[a][i] - scores[b][i] for a, b in PAIRS},
                         'rankings': {m: rankings[m][i] for m in ('cbf', *VARIANTS)},
                         'review': {'status': 'pending_human_review', 'supported_assertions': None,
                                    'important_omissions': None, 'unsupported_assertions': None,
                                    'instruction': 'Compare each assertion to actual input. Distinguish input truncation, harmless omission, and unsupported inference; inspect concentration and repeated-purchase claims.'}})
    # Pick a real, label-bearing reachable case for the student, favoring primary paired change.
    eligible = [i for i in range(200) if truth[i] and set(truth[i]) & set(bundle['candidates'][i])]
    if not eligible:
        eligible = [i for i in range(200) if truth[i]] or list(range(200))
    i = sorted(eligible, key=lambda j: (-abs(scores['summary'][j] - scores['raw'][j]), j))[0]
    def contributions(method):
        return [{'rank': rank, 'ISIN': asset, 'relevant': asset in truth[i], 'discount': DISCOUNTS[rank - 1],
                 'gain': DISCOUNTS[rank - 1] if asset in truth[i] else 0.0}
                for rank, asset in enumerate(rankings[method][i], 1)]
    hand = {'status': 'reserved_unperformed', 'attributed_to_student': False, 'user': f'U{i + 1:03d}', 'position': i + 1,
            'cutoff': '2021-06-01', 'end_exclusive': '2021-07-01',
            'eligible_future_Buy_ISINs': sorted(truth[i]), 'ordered_candidates': bundle['candidates'][i],
            'actual_history_block': bundle['shared'].history_block(bundle['contexts'][i]),
            'actual_summary': inference['stages']['llm-summary'][i]['text'],
            'top10_contributions': {m: contributions(m) for m in ('cbf', *VARIANTS)},
            'ideal_discounts': list(DISCOUNTS[:min(10, len(truth[i]))]),
            'computed_reference': {'idcg': sum(DISCOUNTS[:min(10, len(truth[i]))]),
                                   'ndcg': {m: scores[m][i] for m in ('cbf', *VARIANTS)},
                                   'summary_minus_raw': scores['summary'][i] - scores['raw'][i]},
            'student_worksheet': {'dcg_raw': None, 'dcg_summary': None, 'idcg': None, 'ndcg_raw': None,
                                  'ndcg_summary': None, 'paired_difference': None, 'summary_fidelity_notes': None},
            'instructions': 'Independently mark Buy hits, sum 1/log2(rank+1), divide by IDCG over ALL eligible future Buy assets (including outside the pool); empty truth scores zero. Compare to computed_reference afterward. No student completion is claimed.'}
    return examples, hand

In [ ]:
def render_markdown(summary, examples, hand):
    def f(value):
        return 'n/a' if value is None else f'{value:.4f}'
    lines = ['# RQ2 development evaluation', '',
             f"All {summary['users']} users retained; {summary['nonempty_truth_users']} nonempty truth users; {summary['empty_truth_users']} empty truth users score zero.",
             'History before 2021-06-01; eligible Buy truth in [2021-06-01, 2021-07-01). All frozen starter baselines independently matched.', '',
             '| Method | Users | Nonempty truth | nDCG@10 all | nDCG@10 nonempty | Hit coverage all |',
             '|---|---:|---:|---:|---:|---:|']
    groups = {**summary['starter_baselines'], **summary['matched'],
              'supplied ranker (separate comparison)': summary['supplied_ranker_separate_comparison']}
    for method, row in groups.items():
        lines.append(f"| {method} | {row['users']} | {row['nonempty_truth_users']} | {f(row['ndcg_at_10'])} | {f(row['ndcg_at_10_nonempty'])} | {f(row['hit_coverage_all'])} |")
    lines += ['', '## Paired differences', '', 'Descriptive paired differences; full precision in JSON/CSV. No significance or generalization claim.', '',
              '| Difference / subset | n | Mean delta | Improved | Worsened | Unchanged |', '|---|---:|---:|---:|---:|---:|']
    for pair, subsets in summary['paired'].items():
        for subset, row in subsets.items():
            lines.append(f"| {pair} / {subset} | {row['n']} | {f(row['mean'])} | {row['improved']} | {row['worsened']} | {row['unchanged']} |")
    reach = summary['reachability']
    lines += ['', '## Candidate reachability and failures', '',
              f"Pool reaches truth for {reach['users_with_reachable_truth']} users; {reach['nonempty_users_with_no_reachable_truth']} nonempty users have no reachable truth. Reachable user–truth pairs: {reach['reachable_user_truth_pairs']}/{reach['total_user_truth_pairs']}.",
              f"Pool oracle nDCG@10: {f(reach['oracle_ndcg_all'])} all / {f(reach['oracle_ndcg_nonempty'])} nonempty. Oracle hit coverage: {f(reach['hit_coverage_ceiling_all'])} all / {f(reach['hit_coverage_ceiling_nonempty'])} nonempty. IDCG always includes all eligible truth.", '',
              '| Variant | Repairs | Fallbacks | Rank caps | Context errors | Reachable users with no top-10 hit | Mean oracle gap |',
              '|---|---:|---:|---:|---:|---:|---:|']
    for variant, row in summary['failures'].items():
        lines.append(f"| {variant} | {row['repairs']} | {row['fallbacks']} | {row['token_caps']} | {row['context_errors']} | {row['reachable_users_with_no_top10_hit']} | {f(row['mean_gap_to_pool_oracle'])} |")
    s = summary['summary_generation']
    lines += ['', f"Summary generation: {s['missing_or_empty']} missing/empty, {s['token_cap_count']} capped, {s['context_limit_errors']} context errors. Caps can leave an incomplete summary; they are not automatic hallucination labels.", '',
              '## Changes and failure context', '',
              '| Pair / outcome | Users | Same order | Any fallback | Any repair | Summary cap | Empty truth | Nonempty, no reachable truth |',
              '|---|---:|---:|---:|---:|---:|---:|---:|---:|']
    for pair, groups in summary['changes'].items():
        for outcome, row in groups.items():
            lines.append(f"| {pair} / {outcome} | {row['users']} | {row['same_top10_order']} | {row['any_fallback']} | {row['any_repair']} | {row['summary_capped']} | {row['empty_truth']} | {row['nonempty_no_reachable_truth']} |")
    sparse = summary['sparse_1_to_5_buys']
    lines += ['', '## Sparse history (descriptive)', '',
              f"1–5 eligible pre-cutoff Buy rows: {sparse['users']} users; {sparse['nonempty_truth_users']} nonempty truth; {sparse['users_with_reachable_truth']} reachable. Pool oracle mean: {f(sparse['oracle_ndcg'])}. Coverage and sample size limit interpretation.", '',
              '| Method | nDCG@10 all sparse | nDCG@10 nonempty sparse |', '|---|---:|---:|']
    for method, row in sparse['methods'].items():
        lines.append(f"| {method} | {f(row['ndcg_at_10'])} | {f(row['ndcg_at_10_nonempty'])} |")
    lines += ['', '## Timing and interpretation', '']
    for name, value in summary['timing'].items():
        lines.append(f'- {name}: {f(value) if isinstance(value, (int, float)) else value if value is not None else "unmeasured"}')
    lines += ['', *['- ' + note for note in summary['interpretation']], '',
              '## Audit and student check', '',
              'See [summary-audit.md](summary-audit.md) and [summary-audit.jsonl](summary-audit.jsonl) for actual inputs, summaries and pending review fields.',
              'Audit positions: ' + ', '.join(str(e['position']) for e in examples) + '.',
              'See [failure-examples.jsonl](failure-examples.jsonl) for up to three actual examples per variant and failure flag, including raw outputs and candidate-number mapping. Overlapping flags are retained.',
              f"The real example {hand['user']} is in [student-hand-check.md](student-hand-check.md) and [student-hand-check.json](student-hand-check.json). Its reference was computed by this script; the student worksheet remains unperformed and unattributed.", '',
              'Anonymized labels are frozen cohort positions; no customer ID mapping is exported. Treat these research artifacts as pseudonymized, not a formal anonymization guarantee.']
    return '\n'.join(lines) + '\n'


def write_outputs(bundle, inference, summary, per_user, examples, hand, out=OUT):
    # Defense in depth: replace any source customer identifier in all exported text.
    # IDs are never hashed into a published join key or included in a lookup table.
    identifiers = sorted((str(u) for u in bundle['profiles'] if str(u)), key=len, reverse=True)
    id_pattern = re.compile('|'.join(re.escape(u) for u in identifiers))
    def private_safe(text):
        return id_pattern.sub('[CUSTOMER_ID_REDACTED]', text)
    out.mkdir(parents=True, exist_ok=False)
    def put(name, content):
        (out / name).write_text(private_safe(content), encoding='utf-8')
    def dump(value):
        return json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + '\n'
    def display_json(value):
        # Displayed estimates use four decimals; JSON/CSV artifacts retain precision.
        def rounded(item):
            if isinstance(item, float):
                return round(item, 4)
            if isinstance(item, dict):
                return {k: rounded(v) for k, v in item.items()}
            if isinstance(item, list):
                return [rounded(v) for v in item]
            return item
        return json.dumps(rounded(value), ensure_ascii=False, indent=2)
    def notebook_source_hash(path):
        cells = read_json(path)['cells']
        source = ''.join(''.join(cell['source']) for cell in cells if cell['cell_type'] == 'code')
        return hashlib.sha256(source.encode()).hexdigest()
    inference_path = PROJECT / '.runs/rq2_h100_v2-f4072590ae167d5b/rq2_h100_v2.ipynb'
    if not inference_path.is_file():
        inference_path = PROJECT / 'rq2_h100_v2.ipynb'
    summary['notebook_source_sha256'] = {'inference': notebook_source_hash(inference_path),
                                         'evaluation': notebook_source_hash(PROJECT / 'evaluate_rq2_v2.ipynb')}
    require(summary['notebook_source_sha256']['inference'] ==
            'ae265dcb12781847d899250de556bf9ed11933b91b15be618ef410c0a416b8db',
            'Inference notebook does not match the immutable launch fingerprint')
    summary['source_provenance_note'] = 'Inference source matches the immutable launch receipt; prompts and parser behavior were independently replayed.'
    put('summary.json', dump(summary))
    put('summary.md', render_markdown(summary, examples, hand))
    import io
    csv_buffer = io.StringIO(newline='')
    writer = csv.DictWriter(csv_buffer, fieldnames=list(per_user[0]))
    writer.writeheader()
    writer.writerows(per_user)
    put('per-user.csv', csv_buffer.getvalue())
    put('summary-audit.jsonl', ''.join(json.dumps(e, ensure_ascii=False, allow_nan=False) + '\n' for e in examples))
    audit_lines = ['# Summary audit — pending human review', '',
                   'Check assertions against each actual input. An omitted detail is not by itself a hallucination. The input itself may be capped at 20 assets.', '']
    for example in examples:
        audit_lines += [f"## {example['user']} — position {example['position']}", '',
                        'Selection: ' + ', '.join(example['selection_reasons']), '',
                        'Actual summarizer messages:', '', '```json', json.dumps(example['actual_summary_messages'], ensure_ascii=False, indent=2), '```', '',
                        'Generated summary:', '', '```text', example['generated_summary'] or '[No summary returned.]', '```', '',
                        'Evidence and review flags:', '', '```json', display_json({k: v for k, v in example.items() if k not in ('actual_summary_messages', 'actual_history_block', 'generated_summary')}), '```', '']
    put('summary-audit.md', '\n'.join(audit_lines))
    failure_examples = []
    for variant in (*VARIANTS, 'provided'):
        rows = inference['stages'][variant]
        predictions = inference['stages'][variant + '-predictions']
        for flag in ('fallback', 'repair', 'hit_token_cap', 'context_limit_exceeded'):
            selected = []
            for i, (row, prediction) in enumerate(zip(rows, predictions)):
                active = prediction[flag] if flag in ('fallback', 'repair') else (
                    row['hit_token_cap'] if flag == 'hit_token_cap' else bool(row['error']))
                if active:
                    selected.append(i)
            for i in selected[:3]:
                row, prediction = rows[i], predictions[i]
                failure_examples.append({'user': f'U{i + 1:03d}', 'position': i + 1, 'variant': variant,
                                         'selection_flag': flag, 'raw_rank_output': row['text'],
                                         'ordered_candidates': bundle['candidates'][i],
                                         'final_ranking': prediction['ranking'], 'repair': prediction['repair'],
                                         'fallback': prediction['fallback'], 'error': row['error'],
                                         'hit_token_cap': row['hit_token_cap'],
                                         'truth_ISINs': sorted(bundle['truth'][i]),
                                         'ndcg': bundle['scores'][variant][i]})
    put('failure-examples.jsonl', ''.join(json.dumps(e, ensure_ascii=False, allow_nan=False) + '\n' for e in failure_examples))
    put('student-hand-check.json', dump(hand))
    hand_lines = ['# Real student hand-check worksheet — unperformed', '', f"Example: {hand['user']} (position {hand['position']}).", '', hand['instructions'], '',
                  'Truth and ordered candidates:', '', '```json', json.dumps({k: hand[k] for k in ('eligible_future_Buy_ISINs', 'ordered_candidates')}, indent=2), '```', '',
                  'Actual raw history:', '', '```text', hand['actual_history_block'], '```', '', 'Actual summary:', '', '```text', hand['actual_summary'] or '[No summary returned.]', '```', '',
                  '| Method | Rank | ISIN | Buy hit | Discount | Gain |', '|---|---:|---|---:|---:|---:|']
    for method, rows in hand['top10_contributions'].items():
        for row in rows:
            hand_lines.append(f"| {method} | {row['rank']} | {row['ISIN']} | {int(row['relevant'])} | {row['discount']:.4f} | {row['gain']:.4f} |")
    hand_lines += ['', 'Student worksheet (fill independently):', '', '```json', json.dumps(hand['student_worksheet'], indent=2), '```', '',
                   '<details><summary>Script-computed reference (not a student attribution)</summary>', '', '```json', display_json(hand['computed_reference']), '```', '', '</details>', '']
    put('student-hand-check.md', '\n'.join(hand_lines))
    artifacts = {p.name: sha(p) for p in out.iterdir() if p.is_file()}
    put('artifact-manifest.json', dump(artifacts))

In [ ]:
def main():
    started = time.perf_counter()
    require(not OUT.exists(), 'Evaluation output already exists; preserve or rename it before an intentional rerun')
    bundle = verify_inputs()
    inference = verify_inference(bundle)
    summary, per_user = analyze(bundle, inference)
    examples, hand = prepare_audits(bundle, inference, per_user)
    summary['timing']['evaluation_before_output_seconds'] = time.perf_counter() - started
    write_outputs(bundle, inference, summary, per_user, examples, hand)
    # Safe console output: aggregates/positions only, no prompts, records, or identifiers.
    print(f"Verified and evaluated {summary['users']} users ({summary['nonempty_truth_users']} nonempty truth).")
    print('Reports written to results/rq2_h100_v2/evaluation/. Human audits and student worksheet remain pending.')


if __name__ == '__main__':
    main()